In [15]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

WindowsPath('E:/My_Github/epilepsy-eeg-hybrid-ai')

In [2]:
required_directories = [
    "config",
    "data",
    "metadata",
    "notebooks",
    "results",
    "scripts",
    "src",
    "tests",
]

for directory in required_directories:
    path = PROJECT_ROOT / directory
    print(f"{directory:15} exists={path.exists()}")

config          exists=True
data            exists=True
metadata        exists=True
notebooks       exists=True
results         exists=True
scripts         exists=True
src             exists=True
tests           exists=True


In [29]:
import mne
import numpy as np
import pandas as pd
import sklearn

print("MNE:", mne.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

MNE: 1.12.1
NumPy: 2.4.6
Pandas: 3.0.3
Scikit-learn: 1.9.0


In [9]:
duplicate_comparison = pd.read_csv(
    "metadata/chbmit_duplicate_channel_comparison.csv"
)

duplicate_comparison.sort_values(
    "pearson_correlation",
    ascending=False,
).head(30)

,patient_id,recording_id,relative_path,duplicate_base,first_channel,second_channel,samples_compared,exactly_identical,numerically_allclose,pearson_correlation,maximum_absolute_difference,root_mean_square_difference
4297,chb24,chb24_22,chb24/chb24_22.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
0,chb01,chb01_01,chb01/chb01_01.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
1,chb01,chb01_02,chb01/chb01_02.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
2,chb01,chb01_03,chb01/chb01_03.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
4281,chb24,chb24_06,chb24/chb24_06.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
4280,chb24,chb24_05,chb24/chb24_05.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
4279,chb24,chb24_04,chb24/chb24_04.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
4278,chb24,chb24_03,chb24/chb24_03.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
4277,chb24,chb24_02,chb24/chb24_02.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0
4276,chb24,chb24_01,chb24/chb24_01.edf,T8-P8,T8-P8-0,T8-P8-1,76800,True,True,1.0,0.0,0.0


# Phase 4
# Sanity check of values

In [10]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
QC_DIR = PROJECT_ROOT / "metadata" / "signal_quality"

chunks = pd.read_csv(
    QC_DIR / "chbmit_qc_chunks.csv"
)

channels = pd.read_csv(
    QC_DIR / "chbmit_qc_channels.csv"
)

recordings_qc = pd.read_csv(
    QC_DIR / "chbmit_qc_recordings.csv"
)

print(chunks.shape)
print(channels.shape)
print(recordings_qc.shape)

(1379528, 66)
(17175, 25)
(657, 11)


# Domain Unit Review

In [11]:
chunks[
    [
        "mean_uv",
        "std_uv",
        "rms_uv",
        "peak_to_peak_uv",
        "robust_range_01_99_uv",
    ]
].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

,mean_uv,std_uv,rms_uv,peak_to_peak_uv,robust_range_01_99_uv
count,1.379528e+06,1.379528e+06,1.379528e+06,1.379528e+06,1.379528e+06
mean,1.582809e-01,4.410168e+01,4.415786e+01,5.317822e+02,2.433208e+02
std,4.799045e-01,4.390655e+01,4.385297e+01,5.102728e+02,2.456404e+02
min,-1.711943e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1%,-1.000095e+00,0.000000e+00,1.953602e-01,0.000000e+00,0.000000e+00
5%,-1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
50%,1.953602e-01,3.291204e+01,3.291367e+01,3.997070e+02,1.769260e+02
95%,6.996846e-01,1.088844e+02,1.088901e+02,1.404249e+03,6.293998e+02
99%,1.442504e+00,2.176980e+02,2.176981e+02,2.609231e+03,1.212656e+03
max,1.116506e+01,1.017569e+03,1.017571e+03,9.830525e+03,5.569416e+03


# Finite value check

In [12]:
chunks[
    [
        "nonfinite_fraction",
        "nan_fraction",
        "positive_infinity_fraction",
        "negative_infinity_fraction",
    ]
].max()

nonfinite_fraction            0.0
nan_fraction                  0.0
positive_infinity_fraction    0.0
negative_infinity_fraction    0.0
dtype: float64

# Check annotation tag

In [13]:
chunks["annotation_label"].value_counts()

annotation_label
non_ictal    1369446
boundary        5158
ictal           4924
Name: count, dtype: int64

# Phase 5

In [14]:
import pandas as pd

manifest = pd.read_csv(
    "metadata/harmonization/"
    "chbmit_recording_inclusion_manifest.csv"
)

excluded_seizures = manifest.loc[
    ~manifest["include_primary_analysis"],
    [
        "case_id",
        "recording_id",
        "has_seizure",
        "n_seizures_parsed",
        "total_ictal_seconds",
        "inclusion_reason",
    ],
]

excluded_seizures.loc[
    excluded_seizures["has_seizure"]
]

,case_id,recording_id,has_seizure,n_seizures_parsed,total_ictal_seconds,inclusion_reason
324,chb12,chb12_27,True,6,229.0,missing_primary_channels:C3-P3|C4-P4|CZ-PZ|F3-...
325,chb12,chb12_28,True,1,34.0,missing_primary_channels:C3-P3|C4-P4|CZ-PZ|F3-...
326,chb12,chb12_29,True,6,223.0,missing_primary_channels:C3-P3|C4-P4|CZ-PZ|F3-...


# Real experiment on multiple EDFs
* A normal file
* A seizure-containing file
* A file with extra channels
* A file with different order
* A file close to incomplete pattern

In [15]:
from pathlib import Path

import mne
import pandas as pd
import yaml

from src.channels.harmonize import (
    harmonize_raw_channels,
)
from src.channels.naming import (
    load_validated_aliases,
)

PROJECT_ROOT = Path.cwd()

with (
    PROJECT_ROOT
    / "config"
    / "chbmit_channel_harmonization.yaml"
).open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

target_channels = config[
    "channel_sets"
]["primary_17"]["channels"]

manifest = pd.read_csv(
    PROJECT_ROOT
    / "metadata"
    / "harmonization"
    / "chbmit_recording_inclusion_manifest.csv"
)

alias_table = pd.read_csv(
    PROJECT_ROOT
    / "config"
    / "chbmit_channel_aliases.csv"
)

alias_rules = load_validated_aliases(
    alias_table
)

selected = manifest.loc[
    manifest["include_primary_analysis"]
].iloc[0]

edf_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "chbmit"
    / selected["edf_relative_path"]
)

raw = mne.io.read_raw_edf(
    edf_path,
    preload=False,
    infer_types=False,
    verbose="ERROR",
)

harmonized = harmonize_raw_channels(
    raw=raw,
    target_channels=target_channels,
    recording_id=selected["recording_id"],
    alias_rules=alias_rules,
    copy=True,
)

assert harmonized.ch_names == target_channels
assert raw.ch_names != harmonized.ch_names or (
    len(raw.ch_names) == len(harmonized.ch_names)
)

harmonized

<RawEDF | chb01_01.edf, 17 x 921600 (3600.0 s), ~18 KiB, data not loaded>

# Phase 7

# Manual control of boundary windows

In [16]:
selected = windows.loc[
    windows["label_name"]
    == "boundary"
].iloc[0]

selected[
    [
        "case_id",
        "recording_id",
        "start_seconds",
        "end_seconds",
        "seizure_overlap_seconds",
        "seizure_overlap_fraction",
    ]
]

NameError: name 'windows' is not defined

Reading FIF:

In [ ]:
raw = mne.io.read_raw_fif(
    PROJECT_ROOT
    / selected["output_fif_path"],
    preload=False,
    verbose="ERROR",
)

raw.plot(
    start=max(
        0,
        float(selected["start_seconds"])
        - 4
    ),
    duration=12,
    n_channels=17,
    scalings="auto",
)

# Manual Window Ictal Control

In [ ]:
selected = windows.loc[
    windows["label_name"] == "ictal"
].sort_values(
    "seizure_overlap_fraction",
    ascending=False,
).iloc[0]

# Phase 9
Usage example for Outer 0 / Inner 0

In [13]:
from pathlib import Path

import pandas as pd
import torch
import yaml

from src.data.dataloaders import (
    build_eeg_dataloader,
)
from src.data.fold_selection import (
    get_nested_window_tables,
)
from src.normalization.fold_scalers import (
    load_scaler_npz,
)
from src.splitting.subject_mapping import (
    apply_subject_mapping,
)
from src.utils.reproducibility import (
    seed_everything,
)

PROJECT_ROOT = Path.cwd().parent

Reading config:

In [16]:
with (
    PROJECT_ROOT
    / "config"
    / "chbmit_data_pipeline.yaml"
).open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

Reading data:

In [17]:
windows = pd.read_csv(
    PROJECT_ROOT
    / config["inputs"]["window_manifest"]
)

mapping = pd.read_csv(
    PROJECT_ROOT
    / config["inputs"]["subject_mapping"]
)

windows = apply_subject_mapping(
    windows,
    mapping,
)

inner_assignments = pd.read_csv(
    PROJECT_ROOT
    / config["inputs"]["inner_subject_folds"]
)

Select fold:

In [18]:
fold_tables = get_nested_window_tables(
    windows=windows,
    inner_assignments=inner_assignments,
    outer_fold=0,
    inner_fold=0,
)

for role, table in fold_tables.items():
    print(
        role,
        table["subject_id"].nunique(),
        len(table),
        table["binary_label"]
        .value_counts()
        .to_dict(),
    )

train 13 893901 {0: 890536, 1: 3365}
validation 4 415536 {0: 414423, 1: 1113}
test 5 314864 {0: 313646, 1: 1218}


Scaler

In [19]:
scaler = load_scaler_npz(
    PROJECT_ROOT
    / "artifacts"
    / "scalers"
    / "outer_00"
    / "inner_00_scaler.npz"
)

channel_mean_uv = scaler[
    "channel_mean_uv"
]

channel_std_uv = scaler[
    "channel_std_uv"
]

Seed:

In [20]:
generator = seed_everything(
    seed=42,
    deterministic_algorithms=True,
    warn_only=True,
    cudnn_deterministic=True,
    cudnn_benchmark=False,
)

Train loader:

In [21]:
train_loader = build_eeg_dataloader(
    windows=fold_tables["train"],
    project_root=PROJECT_ROOT,
    channel_mean_uv=channel_mean_uv,
    channel_std_uv=channel_std_uv,
    batch_size=64,
    role="train",
    imbalance_strategy="loss_weighting",
    generator=generator,
    expected_channel_count=17,
    expected_sample_count=1024,
    max_open_fif_files=4,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=2,
    drop_last=True,
    return_metadata=True,
)

Validation loader:

In [22]:
validation_loader = build_eeg_dataloader(
    windows=fold_tables["validation"],
    project_root=PROJECT_ROOT,
    channel_mean_uv=channel_mean_uv,
    channel_std_uv=channel_std_uv,
    batch_size=128,
    role="validation",
    imbalance_strategy="none",
    generator=generator,
    expected_channel_count=17,
    expected_sample_count=1024,
    max_open_fif_files=4,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=2,
    drop_last=False,
    return_metadata=True,
)

Test loader:

In [23]:
test_loader = build_eeg_dataloader(
    windows=fold_tables["test"],
    project_root=PROJECT_ROOT,
    channel_mean_uv=channel_mean_uv,
    channel_std_uv=channel_std_uv,
    batch_size=128,
    role="test",
    imbalance_strategy="none",
    generator=generator,
    expected_channel_count=17,
    expected_sample_count=1024,
    max_open_fif_files=4,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    prefetch_factor=2,
    drop_last=False,
    return_metadata=True,
)

Checking a batch:

In [24]:
signals, labels, metadata = next(
    iter(train_loader)
)

print(signals.shape)
print(labels.shape)
print(signals.dtype)
print(labels.dtype)
print(labels.unique())

C:\Users\Sara\anaconda3\envs\epilepsy-eeg\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([64, 17, 1024])
torch.Size([64])
torch.float32
torch.float32
tensor([0.])


# Making the right loss
Weight corresponding to outer 0 / inner 0:

In [25]:
class_weights = pd.read_csv(
    PROJECT_ROOT
    / "metadata"
    / "data_pipeline"
    / "chbmit_fold_class_weights.csv"
)

fold_weight = class_weights.loc[
    (class_weights["scope"] == "inner_training")
    & (class_weights["outer_fold"] == 0)
    & (class_weights["inner_fold"] == 0),
    "pos_weight",
].iloc[0]

Making loss:

In [26]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [fold_weight],
        dtype=torch.float32,
        device=device,
    )
)

In training:

In [27]:
logits = model(
    signals.to(device)
).squeeze(-1)

loss = criterion(
    logits,
    labels.to(device),
)

NameError: name 'model' is not defined

# Checking the distribution after normalization

In [30]:
channel_sum = np.zeros(17)
channel_squared_sum = np.zeros(17)
sample_count = 0

for batch_index, (
    signals,
    labels,
    metadata,
) in enumerate(train_loader):
    values = signals.numpy()

    channel_sum += values.sum(
        axis=(0, 2)
    )

    channel_squared_sum += (
        values * values
    ).sum(axis=(0, 2))

    sample_count += (
        values.shape[0]
        * values.shape[2]
    )

    if batch_index >= 99:
        break

mean = (
    channel_sum
    / sample_count
)

variance = (
    channel_squared_sum
    / sample_count
    - mean * mean
)

std = np.sqrt(variance)

print("Approximate means:")
print(mean)

print("Approximate stds:")
print(std)

C:\Users\Sara\anaconda3\envs\epilepsy-eeg\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Approximate means:
[ 3.22837422e-04 -1.34546802e-03  3.35633662e-04 -2.51051322e-04
  7.73839619e-05 -8.63699989e-04 -4.63277142e-04  1.27582892e-04
 -4.14716184e-04 -9.45434529e-05 -7.92249091e-04 -1.61309684e-04
 -4.75810423e-04  4.27032927e-04 -5.34959718e-04 -9.07644911e-04
  7.63276157e-04]
Approximate stds:
[0.98155672 0.98394074 0.9846984  1.0268041  0.98355343 0.99762272
 0.97569235 1.01902761 1.00501526 0.99772886 0.99466368 0.99478848
 0.96862989 0.96511694 0.98428292 0.98137864 0.95995916]
